In [1]:
import pandas as pd

X_raw_train = pd.read_csv('../data/processed/X_raw_train.csv', index_col=0)
X_raw_test  = pd.read_csv('../data/processed/X_raw_test.csv',  index_col=0)
X_ext_train = pd.read_csv('../data/processed/X_ext_train.csv', index_col=0)
X_ext_test  = pd.read_csv('../data/processed/X_ext_test.csv',  index_col=0)
y_train     = pd.read_csv('../data/processed/y_train.csv',     index_col=0).squeeze()
y_test      = pd.read_csv('../data/processed/y_test.csv',      index_col=0).squeeze()

print("raw:", X_raw_train.shape, X_raw_test.shape)
print("ext:", X_ext_train.shape, X_ext_test.shape)
print("y:  ", y_train.shape, y_test.shape)
print(y_train.value_counts(normalize=True).round(3))

raw: (9668, 14) (2417, 14)
ext: (9668, 19) (2417, 19)
y:   (9668,) (2417,)
Disease_encoded
4    0.236
0    0.176
5    0.153
2    0.147
1    0.133
6    0.112
3    0.043
Name: proportion, dtype: float64


In [2]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

disease_names = ['Ankylosing Spondylitis', 'Normal', 'Psoriatic Arthritis',
                  'Reactive Arthritis', 'Rheumatoid Arthritis',
                  "Sjögren's Syndrome", 'Systemic Lupus Erythematosus']

def evaluate_model(y_true, y_pred, model_name):
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    weighted_f1 = f1_score(y_true, y_pred, average='weighted')

    print(f"=== {model_name} ===")
    print(f"Macro F1:    {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=disease_names, digits=3))

    cm = confusion_matrix(y_true, y_pred)
    return {'model': model_name, 'macro_f1': macro_f1, 'weighted_f1': weighted_f1,
            'y_true': y_true, 'y_pred': y_pred, 'confusion_matrix': cm}

In [3]:


# multi_class handling: with 7 disease classes, LogisticRegression needs to decide
# how to extend binary logistic regression to multiclass. 'multinomial' fits one
# unified model that directly estimates probabilities across all 7 classes at once
# (more statistically correct than the older one-vs-rest default).
# class_weight='balanced' matters a lot here: it upweights the loss contribution of
# rare classes (Reactive Arthritis, 4.3% of data) so the model doesn't just learn to
# ignore them to minimize overall error — same spirit as why we use macro-F1 for scoring.
# max_iter raised from the sklearn default (100) because multinomial logistic regression
# on 9,668 rows with 14 features often needs more iterations to converge.
from sklearn.linear_model import LogisticRegression

lr_raw = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_raw.fit(X_raw_train, y_train)

y_pred_lr_raw = lr_raw.predict(X_raw_test)
results_lr_raw = evaluate_model(y_test, y_pred_lr_raw, "Logistic Regression (raw features)")

=== Logistic Regression (raw features) ===
Macro F1:    0.7949
Weighted F1: 0.8005

                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.622     0.565     0.592       425
                      Normal      0.882     0.841     0.861       321
         Psoriatic Arthritis      0.805     0.821     0.813       357
          Reactive Arthritis      0.487     0.932     0.640       103
        Rheumatoid Arthritis      0.862     0.775     0.816       570
          Sjögren's Syndrome      0.864     0.878     0.871       370
Systemic Lupus Erythematosus      0.964     0.978     0.971       271

                    accuracy                          0.799      2417
                   macro avg      0.784     0.827     0.795      2417
                weighted avg      0.810     0.799     0.800      2417



In [4]:
lr_ext = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_ext.fit(X_ext_train, y_train)

y_pred_lr_ext = lr_ext.predict(X_ext_test)
results_lr_ext = evaluate_model(y_test, y_pred_lr_ext, "Logistic Regression (extended features)")

=== Logistic Regression (extended features) ===
Macro F1:    0.7921
Weighted F1: 0.7999

                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.624     0.567     0.594       425
                      Normal      0.871     0.822     0.846       321
         Psoriatic Arthritis      0.815     0.838     0.826       357
          Reactive Arthritis      0.468     0.922     0.621       103
        Rheumatoid Arthritis      0.870     0.772     0.818       570
          Sjögren's Syndrome      0.853     0.878     0.866       370
Systemic Lupus Erythematosus      0.974     0.974     0.974       271

                    accuracy                          0.798      2417
                   macro avg      0.782     0.825     0.792      2417
                weighted avg      0.811     0.798     0.800      2417



In [5]:
from sklearn.ensemble import RandomForestClassifier

# n_estimators: number of trees in the forest. More trees = more stable averaging,
# with diminishing returns past a few hundred. 300 is a reasonable, unremarkable baseline —
# no tuning yet, that comes in a later stage.
# max_depth left at default (None = trees grow until leaves are pure or too small to split).
# This sounds reckless, but bagging is exactly what protects against the overfitting
# this would normally cause in a single tree.
# n_jobs=-1: use all available CPU cores to train trees in parallel (they're independent
# of each other, so this is free parallelism, not an approximation).
rf_raw = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                 random_state=42, n_jobs=-1)
rf_raw.fit(X_raw_train, y_train)

y_pred_rf_raw = rf_raw.predict(X_raw_test)
results_rf_raw = evaluate_model(y_test, y_pred_rf_raw, "Random Forest (raw features)")

=== Random Forest (raw features) ===
Macro F1:    0.8282
Weighted F1: 0.8307

                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.769     0.541     0.635       425
                      Normal      0.875     0.826     0.849       321
         Psoriatic Arthritis      0.828     0.916     0.870       357
          Reactive Arthritis      0.646     0.816     0.721       103
        Rheumatoid Arthritis      0.814     0.900     0.855       570
          Sjögren's Syndrome      0.851     0.908     0.878       370
Systemic Lupus Erythematosus      1.000     0.978     0.989       271

                    accuracy                          0.836      2417
                   macro avg      0.826     0.841     0.828      2417
                weighted avg      0.836     0.836     0.831      2417



In [6]:
import pandas as pd

importances = pd.Series(rf_raw.feature_importances_, index=X_raw_train.columns).sort_values(ascending=False)
print(importances)

ESR           0.233531
CRP           0.147576
C3            0.118890
RF            0.113313
Anti-CCP      0.110602
C4            0.104084
HLA-B27       0.039111
Age           0.034155
ANA           0.025522
Anti-Ro       0.024160
Anti-La       0.023919
Anti-Sm       0.009316
Anti-dsDNA    0.009303
Gender        0.006517
dtype: float64


In [7]:
rf_ext = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                 random_state=42, n_jobs=-1)
rf_ext.fit(X_ext_train, y_train)

y_pred_rf_ext = rf_ext.predict(X_ext_test)
results_rf_ext = evaluate_model(y_test, y_pred_rf_ext, "Random Forest (extended features)")

=== Random Forest (extended features) ===
Macro F1:    0.8274
Weighted F1: 0.8307

                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.744     0.560     0.639       425
                      Normal      0.861     0.847     0.854       321
         Psoriatic Arthritis      0.832     0.902     0.866       357
          Reactive Arthritis      0.638     0.806     0.712       103
        Rheumatoid Arthritis      0.822     0.889     0.854       570
          Sjögren's Syndrome      0.864     0.892     0.878       370
Systemic Lupus Erythematosus      1.000     0.978     0.989       271

                    accuracy                          0.835      2417
                   macro avg      0.823     0.839     0.827      2417
                weighted avg      0.833     0.835     0.831      2417



In [8]:
import os

os.makedirs('../results', exist_ok=True)

def log_result(results_dict):
    return {
        'model': results_dict['model'],
        'macro_f1': round(results_dict['macro_f1'], 4),
        'weighted_f1': round(results_dict['weighted_f1'], 4),
    }

stage1_log = pd.DataFrame([
    log_result(results_lr_raw),
    log_result(results_lr_ext),
    log_result(results_rf_raw),
    log_result(results_rf_ext),
])

stage1_log.to_csv('../results/stage1_summary.csv', index=False)
stage1_log

,model,macro_f1,weighted_f1
0,Logistic Regression (raw features),0.7949,0.8005
1,Logistic Regression (extended features),0.7921,0.7999
2,Random Forest (raw features),0.8282,0.8307
3,Random Forest (extended features),0.8274,0.8307


In [9]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [10]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

xgb_raw = XGBClassifier(
    n_estimators=300,      # boosting rounds — same scale as RF's tree count, different role
    max_depth=5,           # shallow on purpose — see table above
    learning_rate=0.1,     # shrinkage: how much each tree's correction counts (lower = more conservative, needs more rounds)
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
xgb_raw.fit(X_raw_train, y_train, sample_weight=sample_weights)

y_pred_xgb_raw = xgb_raw.predict(X_raw_test)
results_xgb_raw = evaluate_model(y_test, y_pred_xgb_raw, "XGBoost (raw features)")

=== XGBoost (raw features) ===
Macro F1:    0.8280
Weighted F1: 0.8319

                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.720     0.600     0.655       425
                      Normal      0.841     0.857     0.849       321
         Psoriatic Arthritis      0.842     0.908     0.873       357
          Reactive Arthritis      0.643     0.786     0.707       103
        Rheumatoid Arthritis      0.836     0.865     0.850       570
          Sjögren's Syndrome      0.873     0.873     0.873       370
Systemic Lupus Erythematosus      1.000     0.978     0.989       271

                    accuracy                          0.834      2417
                   macro avg      0.822     0.838     0.828      2417
                weighted avg      0.833     0.834     0.832      2417



In [11]:
stage1_log = pd.concat([stage1_log, pd.DataFrame([log_result(results_xgb_raw)])], ignore_index=True)
stage1_log.to_csv('../results/stage1_summary.csv', index=False)
stage1_log

,model,macro_f1,weighted_f1
0,Logistic Regression (raw features),0.7949,0.8005
1,Logistic Regression (extended features),0.7921,0.7999
2,Random Forest (raw features),0.8282,0.8307
3,Random Forest (extended features),0.8274,0.8307
4,XGBoost (raw features),0.8280,0.8319


In [12]:
xgb_ext = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                         objective='multi:softprob', eval_metric='mlogloss',
                         random_state=42, n_jobs=-1)
xgb_ext.fit(X_ext_train, y_train, sample_weight=sample_weights)

y_pred_xgb_ext = xgb_ext.predict(X_ext_test)
results_xgb_ext = evaluate_model(y_test, y_pred_xgb_ext, "XGBoost (extended features)")

=== XGBoost (extended features) ===
Macro F1:    0.8304
Weighted F1: 0.8324

                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.721     0.595     0.652       425
                      Normal      0.855     0.844     0.850       321
         Psoriatic Arthritis      0.850     0.908     0.878       357
          Reactive Arthritis      0.661     0.796     0.722       103
        Rheumatoid Arthritis      0.825     0.870     0.847       570
          Sjögren's Syndrome      0.865     0.881     0.873       370
Systemic Lupus Erythematosus      1.000     0.982     0.991       271

                    accuracy                          0.835      2417
                   macro avg      0.825     0.839     0.830      2417
                weighted avg      0.833     0.835     0.832      2417



In [13]:
stage1_log = pd.concat([stage1_log, pd.DataFrame([log_result(results_xgb_ext)])], ignore_index=True)
stage1_log.to_csv('../results/stage1_summary.csv', index=False)
stage1_log

,model,macro_f1,weighted_f1
0,Logistic Regression (raw features),0.7949,0.8005
1,Logistic Regression (extended features),0.7921,0.7999
2,Random Forest (raw features),0.8282,0.8307
3,Random Forest (extended features),0.8274,0.8307
4,XGBoost (raw features),0.8280,0.8319
5,XGBoost (extended features),0.8304,0.8324
